In [1]:
import os.path as op
import numpy as np
import pandas as pd
from mne.io import read_epochs_eeglab
import scipy.io
from superlet import superlet, scale_from_period
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from burst_detection import extract_bursts
from fooof import FOOOF
from fooof.sim.gen import gen_aperiodic
import pickle

In [2]:
pipeline='NEARICA_behav_v3'
epoch_types=['eye','task']

In [3]:
subjects = pd.read_csv('/home/bonaiuto/dcd_bursts/data/participants.tsv', sep='\t')

subject_ids = subjects['participant_id']

max_freq=50
num=100
foi = np.linspace(1, max_freq, num)
scales = scale_from_period(1/foi)

beta_lims = [14, 31.25]
search_range = np.where((foi>=(beta_lims[0]-3)) & (foi<=(beta_lims[1]+3)))[0]

def process_cluster(subj_id, epoch_type, cluster, cluster_chans):
    subject_data_dir = op.join('/home/bonaiuto/dcd_bursts/data/derivatives', pipeline, subj_id, 'processed_data')
    fname = f'{subj_id}_{epoch_type}.set'
    fpath = op.join(subject_data_dir, fname)

    if not op.exists(fpath):
        return

    EEG = read_epochs_eeglab(fpath)
    EEG_data = EEG.get_data()
    
    bursts={
        'subject': [],
        'epoch_type': [],
        'condition': [],
        'channel': [],
        'trial': [],
        'waveform': np.zeros((0,130)),
        'waveform_times': [],
        'peak_freq': [],
        'peak_amp_iter': [],
        'peak_amp_base': [],
        'peak_time': [],
        'peak_adjustment': [],
        'fwhm_freq': [],
        'fwhm_time': [],
        'polarity': [],
    }
    cond_beta_pow={}
    cond_tf={}
    for cluster_chan in cluster_chans:
        ch_idx = np.where(np.in1d(EEG.info.ch_names, cluster_chan))[0][0]
        clus_data = EEG_data[:, ch_idx, :]
        tf_array = np.zeros((clus_data.shape[0], len(foi), clus_data.shape[1]), dtype=np.single)

        def do_superlet(signal, scales, sfreq):
            spec = superlet(
                signal,
                samplerate=sfreq,
                scales=scales,
                order_max=40,
                order_min=1,
                c_1=4,
                adaptive=True,
            )
            return np.single(np.abs(spec))

        for ix in range(clus_data.shape[0]):
            tf_array[ix, :, :] = do_superlet(clus_data[ix, :], scales, EEG.info['sfreq'])

        psd = np.squeeze(np.mean(np.mean(tf_array, axis=-1), axis=0))
        ff = FOOOF()
        ff.fit(foi, psd, [foi[0], 50])
        ap_params = ff.aperiodic_params_
        ap_fit_log = gen_aperiodic(foi, ap_params, ff.aperiodic_mode)
        ap_fit = 10 ** ap_fit_log
        fooof_THR = ap_fit[search_range].reshape(-1, 1)

        plt.figure(figsize=(8,3))
        plt.subplot(1,2,1)
        plt.plot(foi, np.log10(psd))
        plt.plot(foi, np.log10(ap_fit))
        plt.axvline(beta_lims[0], color='k', linestyle='--')
        plt.axvline(beta_lims[1], color='k', linestyle='--')
        plt.title(f'{subj_id} - {epoch_type} - {cluster}')

        plt.subplot(1,2,2)
        m_tf=np.mean(tf_array[:,search_range,:]-fooof_THR,axis=0)
        b_idx=(EEG.times>-4) & (EEG.times<-3)
        if len(np.where(b_idx)[0]):
            b_tf=np.mean(m_tf[:,b_idx],axis=-1)
            m_tf=(m_tf-b_tf[:,np.newaxis])/b_tf[:,np.newaxis]*100
        plt.imshow(
            m_tf,
            aspect='auto',
            origin='lower',
            extent=[EEG.times[0], EEG.times[-1], foi[search_range[0]], foi[search_range[-1]]]
        )
        fig_fname=op.join('/home/bonaiuto/dcd_bursts/data/derivatives', pipeline, subj_id, f'10_{cluster}_superlets.png')
        plt.savefig(fig_fname)

        chan_bursts = extract_bursts(
            clus_data,
            tf_array[:, search_range, :],
            EEG.times,
            foi[search_range],
            beta_lims,
            fooof_THR,
            EEG.info['sfreq'],
            w_size=0.26,
            regress_evoked=False
        )

        n_bursts = len(chan_bursts['trial'])
        chan_bursts['cluster'] = np.array([cluster] * n_bursts)
        chan_bursts['channel'] = np.array([cluster_chan] * n_bursts)
        chan_bursts['epoch_type'] = np.array([epoch_type] * n_bursts)
        chan_bursts['subject'] = np.array([subj_id] * n_bursts)
        ivd = {v: k for k, v in EEG.event_id.items()}
        chan_bursts['condition'] = np.array([ivd[key] for key in EEG.events[chan_bursts['trial'], 2]])

        for key in bursts.keys():
            if key=='waveform_times':
                bursts[key] = chan_bursts[key]
            elif key=='waveform':
                bursts[key] = np.vstack([bursts[key], chan_bursts[key]])
            else:
                bursts[key] = np.hstack([bursts[key], chan_bursts[key]])

        beta_pow = np.mean(tf_array[:,(foi>=beta_lims[0]) & (foi<=beta_lims[1]),:],axis=1)
        for k, v in EEG.event_id.items():
            if not k in cond_beta_pow:
                cond_beta_pow[k]=[]
                cond_tf[k]=[]
            trial_idx=EEG.events[:,2]==v
            cond_beta_pow[k].append(beta_pow[trial_idx,:])
            cond_tf[k].append(np.mean(tf_array[trial_idx,:,:],axis=0))
            
    output_file = op.join(subject_data_dir, f'bursts_{subj_id}_{epoch_type}_{cluster}.pickle')
    with open(output_file, "wb") as fp:
        pickle.dump(bursts, fp)
    
    output_file = op.join(subject_data_dir, f'beta_pow_{subj_id}_{epoch_type}_{cluster}.pickle')
    with open(output_file, "wb") as fp:
        pickle.dump(cond_beta_pow, fp)
        
    output_file = op.join(subject_data_dir, f'tf_{subj_id}_{epoch_type}_{cluster}.npz')
    np.savez(
        output_file,
        subj_id=subj_id,
        cond_tf=cond_tf,
        time=EEG.times,
        freqs=foi
    )

cluster_chans={
    'C3': ['C3'],
    'C4': ['C4']
}
# --- Parallel Execution ---
Parallel(n_jobs=-1)(
    delayed(process_cluster)(subj_id, epoch_type, cluster, cluster_chans[cluster])
    for subj_id in subject_ids
    for epoch_type in epoch_types
    for cluster in ['C3', 'C4']
)

/home/bonaiuto/miniconda3/envs/dev_beta_umd/lib/python3.9/site-packages/mne/datasets/eegbci/eegbci.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/bonaiuto/miniconda3/envs/dev_beta_umd/lib/python3.9/site-packages/mne/datasets/eegbci/eegbci.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/bonaiuto/miniconda3/envs/dev_beta_umd/lib/python3.9/site-packages/mne/datasets/eegbci/eegbci.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for re

/home/bonaiuto/miniconda3/envs/dev_beta_umd/lib/python3.9/site-packages/mne/datasets/eegbci/eegbci.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/bonaiuto/miniconda3/envs/dev_beta_umd/lib/python3.9/site-packages/mne/datasets/eegbci/eegbci.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/bonaiuto/miniconda3/envs/dev_beta_umd/lib/python3.9/site-packages/mne/datasets/eegbci/eegbci.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for re

/home/bonaiuto/miniconda3/envs/dev_beta_umd/lib/python3.9/site-packages/mne/datasets/eegbci/eegbci.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/bonaiuto/miniconda3/envs/dev_beta_umd/lib/python3.9/site-packages/mne/datasets/eegbci/eegbci.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/bonaiuto/miniconda3/envs/dev_beta_umd/lib/python3.9/site-packages/mne/datasets/eegbci/eegbci.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for re

Extracting parameters from /home/bonaiuto/dcd_bursts/data/derivatives/NEARICA_behav_v3/C17/processed_data/C17_eye.set...
Extracting parameters from /home/bonaiuto/dcd_bursts/data/derivatives/NEARICA_behav_v3/C17/processed_data/C17_eye.set...
Extracting parameters from /home/bonaiuto/dcd_bursts/data/derivatives/NEARICA_behav_v3/C15/processed_data/C15_eye.set...
Extracting parameters from /home/bonaiuto/dcd_bursts/data/derivatives/NEARICA_behav_v3/C15/processed_data/C15_eye.set...
Not setting metadata
2 matching events found
Extracting parameters from /home/bonaiuto/dcd_bursts/data/derivatives/NEARICA_behav_v3/C03/processed_data/C03_eye.set...
Extracting parameters from /home/bonaiuto/dcd_bursts/data/derivatives/NEARICA_behav_v3/C14/processed_data/C14_task.set...
Extracting parameters from /home/bonaiuto/dcd_bursts/data/derivatives/NEARICA_behav_v3/C07/processed_data/C07_eye.set...
Not setting metadata
Not setting metadata
2 matching events found
2 matching events found
Not setting metad

No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from /home/bonaiuto/dcd_bursts/data/derivatives/NEARICA_behav_v3/C15/processed_data/C15_task.set...
Extracting parameters from /home/bonaiuto/dcd_bursts/data/derivatives/NEARICA_behav_v3/C15/processed_data/C15_task.set...
No baseline correction applied
0 projection items activated
No baseline correction applied
No baseline correction applied
0 projection items activated
0 projection items activated
Ready.
Ready.
Ready.
No baseline correction applied
0 projection items activated
Ready.
No baseline correction applied
0 projection items activated
Ready.
No baseline correction applied
0 projection items activated
Ready.
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from /home/bonaiuto/dcd_bursts/data/derivatives/NEARICA_behav_v3/C16/processed_data/C16_task.set...
Not setting metadata
26 matching events found
No baseline correction applied
0 projection items ac

...

Not setting metadata
94 matching events found
Not setting metadata
94 matching events found

FOOOF WARNING: Lower-bound peak width limit is < or ~= the frequency resolution: 0.49 <= 0.50
	Lower bounds below frequency-resolution have no effect (effective lower bound is the frequency resolution).
	Too low a limit may lead to overfitting noise as small bandwidth peaks.
	We recommend a lower bound of approximately 2x the frequency resolution.

Extracting parameters from /home/bonaiuto/dcd_bursts/data/derivatives/NEARICA_behav_v3/C58/processed_data/C58_eye.set...
Not setting metadata
3 matching events found
No baseline correction applied
0 projection items activated
Ready.

FOOOF WARNING: Lower-bound peak width limit is < or ~= the frequency resolution: 0.49 <= 0.50
	Lower bounds below frequency-resolution have no effect (effective lower bound is the frequency resolution).
	Too low a limit may lead to overfitting noise as small bandwidth peaks.
	We recommend a lower bound of approximately 2


FOOOF WARNING: Lower-bound peak width limit is < or ~= the frequency resolution: 0.49 <= 0.50
	Lower bounds below frequency-resolution have no effect (effective lower bound is the frequency resolution).
	Too low a limit may lead to overfitting noise as small bandwidth peaks.
	We recommend a lower bound of approximately 2x the frequency resolution.


FOOOF WARNING: Lower-bound peak width limit is < or ~= the frequency resolution: 0.49 <= 0.50
	Lower bounds below frequency-resolution have no effect (effective lower bound is the frequency resolution).
	Too low a limit may lead to overfitting noise as small bandwidth peaks.
	We recommend a lower bound of approximately 2x the frequency resolution.


FOOOF WARNING: Lower-bound peak width limit is < or ~= the frequency resolution: 0.49 <= 0.50
	Lower bounds below frequency-resolution have no effect (effective lower bound is the frequency resolution).
	Too low a limit may lead to overfitting noise as small bandwidth peaks.
	We recommend a low


FOOOF WARNING: Lower-bound peak width limit is < or ~= the frequency resolution: 0.49 <= 0.50
	Lower bounds below frequency-resolution have no effect (effective lower bound is the frequency resolution).
	Too low a limit may lead to overfitting noise as small bandwidth peaks.
	We recommend a lower bound of approximately 2x the frequency resolution.


FOOOF WARNING: Lower-bound peak width limit is < or ~= the frequency resolution: 0.49 <= 0.50
	Lower bounds below frequency-resolution have no effect (effective lower bound is the frequency resolution).
	Too low a limit may lead to overfitting noise as small bandwidth peaks.
	We recommend a lower bound of approximately 2x the frequency resolution.


FOOOF WARNING: Lower-bound peak width limit is < or ~= the frequency resolution: 0.49 <= 0.50
	Lower bounds below frequency-resolution have no effect (effective lower bound is the frequency resolution).
	Too low a limit may lead to overfitting noise as small bandwidth peaks.
	We recommend a low

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,


FOOOF WARNING: Lower-bound peak width limit is < or ~= the frequency resolution: 0.49 <= 0.50
	Lower bounds below frequency-resolution have no effect (effective lower bound is the frequency resolution).
	Too low a limit may lead to overfitting noise as small bandwidth peaks.
	We recommend a lower bound of approximately 2x the frequency resolution.


FOOOF WARNING: Lower-bound peak width limit is < or ~= the frequency resolution: 0.49 <= 0.50
	Lower bounds below frequency-resolution have no effect (effective lower bound is the frequency resolution).
	Too low a limit may lead to overfitting noise as small bandwidth peaks.
	We recommend a lower bound of approximately 2x the frequency resolution.


FOOOF WARNING: Lower-bound peak width limit is < or ~= the frequency resolution: 0.49 <= 0.50
	Lower bounds below frequency-resolution have no effect (effective lower bound is the frequency resolution).
	Too low a limit may lead to overfitting noise as small bandwidth peaks.
	We recommend a low